In [1]:
# --- Step 1: 设置环境 ---

# Fork 项目
!git clone https://github.com/Cl0udTide/happy-llm
%cd happy-llm

# 安装所有依赖
!git checkout my-experiments
%cd ./my_experiments/_3_training_pipeline
%pip install -r ./requirements.txt

# --- 登录 SwanLab ---
import swanlab
swanlab.login()

Cloning into 'happy-llm'...
remote: Enumerating objects: 1606, done.
remote: Counting objects: 100% (527/527), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 1606 (delta 409), reused 389 (delta 377), pack-reused 1079 (from 2)
Receiving objects: 100% (1606/1606), 50.47 MiB | 47.76 MiB/s, done.
Resolving deltas: 100% (810/810), done.
/content/happy-llm
Branch 'my-experiments' set up to track remote branch 'my-experiments' from 'origin'.
Switched to a new branch 'my-experiments'
/content/happy-llm/my_experiments/_3_training_pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

swanlab: You can find your API key at: https://swanlab.cn/space/~/settings

swanlab: Paste an API key from your profile and hit enter, or press 'CTRL + C' to quit

:

··········


Output()

In [2]:
# ======================================================
# Step 2: 数据准备
# ======================================================
from datasets import load_dataset
import os

# 创建数据目录
os.makedirs("./data/wikipedia_cn", exist_ok=True)
os.makedirs("./data/alpaca_gpt4_zh", exist_ok=True)

# 下载并保存 Wikipedia 数据
print("正在下载 Wikipedia-CN 数据集...")
full_wiki_dataset = load_dataset("pleisto/wikipedia-cn-20230720-filtered", split="train")
print(f"完整维基百科数据集大小: {len(full_wiki_dataset)} 条")

subset_size = len(full_wiki_dataset) // 6
print(f"将使用部分数据: {subset_size} 条")
# 使用 .select() 方法来创建一个子集
subset_wiki_dataset = full_wiki_dataset.select(range(subset_size))

subset_wiki_dataset.to_json("./data/wikipedia_cn/pretrain_data_subset.jsonl")
print("Wikipedia-CN 的子集保存成功！")


# 下载并保存 Alpaca 数据
print("\n正在下载 Alpaca-GPT4-ZH 数据集...")
alpaca_dataset = load_dataset("c-s-ale/alpaca-gpt4-data-zh", split="train")
print(f"完整 Alpaca-GPT4-ZH 数据集大小: {len(alpaca_dataset)} 条")

subset_size = len(alpaca_dataset) // 4
print(f"将使用部分数据: {subset_size} 条")
# 使用 .select() 方法来创建一个子集
subset_alpaca_dataset = alpaca_dataset.select(range(subset_size))

subset_alpaca_dataset.to_json("./data/alpaca_gpt4_zh/sft_data.jsonl")
print("Alpaca-GPT4-ZH 的子集保存成功！")

正在下载 Wikipedia-CN 数据集...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikipedia-cn-20230720-filtered.json:   0%|          | 0.00/524M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254547 [00:00<?, ? examples/s]

...

alpaca_gpt4_data_zh.json:   0%|          | 0.00/35.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48818 [00:00<?, ? examples/s]

完整 Alpaca-GPT4-ZH 数据集大小: 48818 条
将使用部分数据: 12204 条


Creating json from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Alpaca-GPT4-ZH 的子集保存成功！


In [3]:
# ======================================================
# Step 3: 启动预训练
# ======================================================

!python pretrain.py \
    --out_dir "./outputs/tiny_llama_pretrained_wiki" \
    --data_path "./data/wikipedia_cn/pretrain_data_subset.jsonl" \
    --use_swanlab \
    --epochs 1 \
    --batch_size 4 \
    --accumulation_steps 8 \
    --learning_rate 3e-4 \
    --dtype "float16" \
    --gpus "0"

2025-11-07 02:46:07.415795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762483567.435702     684 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762483567.441738     684 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762483567.456832     684 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762483567.456860     684 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762483567.456864     684 computation_placer.cc:177] computation placer alr

In [4]:
# ======================================================
# Step 4: 启动监督微调 (SFT)
# ======================================================

!python finetune.py \
    --base_model_path "./outputs/tiny_llama_pretrained_wiki/pretrain_256_4_8192.pth" \
    --out_dir "./outputs/tiny_llama_sft_alpaca" \
    --data_path "./data/alpaca_gpt4_zh/sft_data.jsonl" \
    --use_swanlab \
    --epochs 1 \
    --batch_size 2 \
    --accumulation_steps 8 \
    --learning_rate 2e-5 \
    --dtype "float16" \
    --gpus "0"

2025-11-07 02:51:06.538582: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762483866.558759    2098 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762483866.564827    2098 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762483866.580119    2098 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762483866.580145    2098 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762483866.580149    2098 computation_placer.cc:177] computation placer alr

In [5]:
# ======================================================
# Step 5: 评估对比 Pretrain vs. SFT
# ======================================================

# --- 测试 Pretrain 模型 ---
print("--- 正在测试 Pretrain 模型 ---")
!python generate.py \
    --model_path "./outputs/tiny_llama_pretrained_wiki/" \
    --prompt "中国的首都是哪里？"

# --- 测试 SFT 模型 ---
print("\n--- 正在测试 SFT 模型 ---")
# 使用 Chat Template 提问
sft_prompt = "### Instruction:\n中国的首都是哪里？\n\n### Response:\n"
!python generate.py \
    --model_path "./outputs/tiny_llama_sft_alpaca/" \
    --prompt "{sft_prompt}"

--- 正在测试 Pretrain 模型 ---
2025-11-07 02:53:49.992378: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762484030.013126    2904 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762484030.019471    2904 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762484030.035876    2904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762484030.035910    2904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762484030.035914    2904 computation_placer.cc:17